In [2]:
# loading libraries for data manipulation
import numpy as np
import pandas as pd

# loading libraries for data visualization
import matplotlib.pyplot as plt
from plotnine import *

# import tensorflow and keras packages
import tensorflow as tf
from tensorflow import keras

import warnings
warnings.filterwarnings('ignore')

In [1]:
# function will take a NN history and plot the training/validation loss and accuracy per epoch
def evaluation_plots(model_history):
    # Extract metrics from the history object
    history_dict = model_history.history
    loss = history_dict['loss']
    val_loss = history_dict['val_loss']
    accuracy = history_dict['accuracy']
    val_accuracy = history_dict['val_accuracy']

    epochs = range(1, len(loss) + 1)

    # Create a figure with 2 subplots (side-by-side)
    plt.figure(figsize=(14, 5))

    # Plot 1: Loss Curves
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, 'r--', label='Training Loss', linewidth=2)
    plt.plot(epochs, val_loss, 'b-', label='Validation Loss', linewidth=2)
    plt.title('Training and Validation Loss (No Regularization)', fontsize=12)
    plt.xlabel('Epochs', fontsize=10)
    plt.ylabel('Loss', fontsize=10)
    plt.legend(fontsize=10)
    plt.grid(True)

    # Plot 2: Accuracy Curves
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, 'r--', label='Training Accuracy', linewidth=2)
    plt.plot(epochs, val_accuracy, 'b-', label='Validation Accuracy', linewidth=2)
    plt.title('Training and Validation Accuracy (No Regularization)', fontsize=12)
    plt.xlabel('Epochs', fontsize=10)
    plt.ylabel('Accuracy', fontsize=10)
    plt.legend(fontsize=10)
    plt.grid(True)

    plt.tight_layout()
    plt.show()


For this exercise, we will use a new dataset called Fashion MNIST. More details here: https://keras.io/api/datasets/fashion_mnist/ 

In [ ]:
# Load the Fashion MNIST data
(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = keras.datasets.fashion_mnist.load_data()

Visualize some examples from the dataset

In [ ]:
plt.imshow(X_train_mnist[0])

Let's flatten the images and one-hot encode the target. 

In [12]:
X_train_mnist = X_train_mnist.reshape(-1, 28*28).astype('float32') / 255.0
X_test_mnist = X_test_mnist.reshape(-1, 28*28).astype('float32') / 255.0

# Convert y labels to one-hot encoded vectors
y_train_mnist = keras.utils.to_categorical(y_train_mnist, num_classes=10)
y_test_mnist = keras.utils.to_categorical(y_test_mnist, num_classes=10)



Regularization techniques ensure that a deep neural network is generalized, and avoids overfitting in particular. Some techniques we can employ:
- Penalization
- Dropout
- Batch Normalization
- Early Stopping

Let's first build a deep neural network to classify digits without any regularization

In [ ]:
model = keras.Sequential([
        keras.layers.Dense(256,activation='relu',input_shape=(784,)),
        keras.layers.Dense(256,activation='relu'),
        keras.layers.Dense(10,activation='softmax')
    ])

model.compile(loss="categorical_crossentropy",
              optimizer="adam", 
              metrics=["accuracy"])

history_no_reg = model.fit(X_train_mnist,y_train_mnist,
                           epochs=20,verbose=1,batch_size=128,
                           validation_data=(X_test_mnist,y_test_mnist))

In [ ]:
evaluation_plots(history_no_reg)

The overfitting is quite apparent here.

Let's introduce L2 weight regularization. This is theoretically a weight decay which will ensure that weights remain small and not one neuron influences inferences. 

In [ ]:
model = keras.Sequential([
        keras.layers.Dense(256,activation='relu',input_shape=(784,)),
        keras.layers.Dense(256,activation='relu'),
        keras.layers.Dense(10,activation='softmax')
    ])

model.compile(loss="categorical_crossentropy",
              optimizer="adam", 
              metrics=["accuracy"])

history_l2_reg = model.fit(X_train_mnist,y_train_mnist,
                           epochs=20,verbose=1,batch_size=128,
                           validation_data=(X_test_mnist,y_test_mnist))

In [ ]:
evaluation_plots(history_l2_reg)

Let's also introduce Dropout. Dropout with a probability p will randomly drop neurons from network during training. 

In [ ]:
model = keras.Sequential([
        keras.layers.Dense(256,activation='relu',input_shape=(784,)),
        keras.layers.Dense(256,activation='relu'),
        keras.layers.Dense(10,activation='softmax')
    ])

model.compile(loss="categorical_crossentropy",
              optimizer="adam", 
              metrics=["accuracy"])

history_l2_drop_reg = model.fit(X_train_mnist,y_train_mnist,
                           epochs=20,verbose=1,batch_size=128,
                           validation_data=(X_test_mnist,y_test_mnist))

In [ ]:
evaluation_plots(history_l2_drop_reg)

Another option would be to use Early Stopping. This mechanism stops training when validation loss stops improving, preventing overfitting. We can use the EarlyStopping callback. 

In [ ]:
model = keras.Sequential([
        keras.layers.Dense(256,activation='relu',input_shape=(784,)),
        keras.layers.Dense(256,activation='relu'),
        keras.layers.Dense(10,activation='softmax')
    ])

model.compile(loss="categorical_crossentropy",
              optimizer="adam", 
              metrics=["accuracy"])

early_stopping = keras.callbacks.EarlyStopping(
    monitor = "val_loss", # track validation loss
    patience = 2, # wait 2 epochs for improvement
    restore_best_weights = True # use best weights learned as final set
)

history_no_reg_early_stop = model.fit(X_train_mnist,y_train_mnist,
                           epochs=20,verbose=1,batch_size=128,
                           validation_data=(X_test_mnist,y_test_mnist),
                           callbacks=[early_stopping])

In [ ]:
evaluation_plots(history_no_reg_early_stop)

Our last regularization step (not including data augmentation), is Batch Normalization. This normalizes activations of each layer to mean = 0 and std = 1, per mini-batch. It stabilizes and speeds up training. Note that we can apply Batch Normalization before the activation function is applied. That is, normalize the weight sum and then apply an activation function on top. You can apply Batch Normalization after activation function as well, but be careful as ReLU can result in a lot of 0s.  

In [ ]:
model = keras.Sequential([
        keras.layers.Dense(256,activation='relu',input_shape=(784,)),
        keras.layers.Dense(256,activation='relu'),
        keras.layers.Dense(10,activation='softmax') # output layer
    ])

model.compile(loss="categorical_crossentropy",
              optimizer="adam", 
              metrics=["accuracy"])


history_no_reg = model.fit(X_train_mnist,y_train_mnist,
                           epochs=5,verbose=1,batch_size=128,
                           validation_data=(X_test_mnist,y_test_mnist))

In [ ]:
model.summary()

This does mean that the network now has to learn more parameters throughout the network. 

Traditionally, the layout for a feed forward neural network should be
1. Define Input Shape
2. Add N Hidden Layers, and for each, add L2 regularization, Activation Function, Batch Normalization (before/after activation)
3. Add Dropout between layers (after activation)
4. Define Output Layer with task-depended activation function (softmax for classification, sigmoid for binary, none for regression/linear)
5. Define Optimizer, Loss Function
6. Define Early Stopping parameters